The following code helps you visualize the submarine data and contains the crucial information about the data such as size of measurement domain, number of grid points and frequency modes etc.

Our first task is to import the data into python (similarly MATLAB). Three files are available, "subdata.npy" is a numpy binary file most convenient for Python users and "subdata.csv" is the same data in text format if you are having trouble with the .npy format. In Python, and in particular if you are using something like Colab you need some utilities to load the data. Here's a small snippet for loading data from your Google drive.   

In [ ]:
from google.colab import drive

# the following command loads up your google drive. It will prompt you to give Colab access to your Gdrive
drive.mount('/content/drive')

In [ ]:
import numpy as np

data_path = '/content/subdata.npy'

d = np.load(data_path) # load npy file

The variable d is a 262144 x 49 matrix. Each column contains a flattened 3D matrix of size 64x64x64

In [ ]:
# import libraries for plotting isosurfaces
import plotly
import plotly.graph_objs as go
# utility for clearing output of cell as loop runs in notebook
from IPython.display import clear_output

# plot the data in time

# NOTE: L we defined in class is 2Lh here, i.e. the domain here is [-Lh,Lh].
Lh = 10; # length of spatial domain (cube of side L = 2*10).
N_grid = 64; # number of grid points/Fourier modes in each direction
xx = np.linspace(-Lh, Lh, N_grid+1) #spatial grid in x dir
x = xx[0:N_grid]
y = x # same grid in y,z direction
z = x

K_grid = (2*np.pi/(2*Lh))*np.linspace(-N_grid/2, N_grid/2 -1, N_grid) # frequency grid for one coordinate

xv, yv, zv = np.meshgrid( x, y, z) # generate 3D meshgrid for plotting

# plot iso surfaces for every third measurement

for j in range(0,49,3):

  signal = np.reshape(d[:, j], (N_grid, N_grid, N_grid))
  normal_sig_abs = np.abs(signal)/np.abs(signal).max()

  # generate data for isosurface of the 3D data
  fig_data = go.Isosurface( x = xv.flatten(), y = yv.flatten(), z = zv.flatten(),
                           value = normal_sig_abs.flatten(), isomin=0.6, isomax=1)

  # generate plots
  clear_output(wait=True) # need this to discard previous figs
  fig = go.Figure( data = fig_data )
  fig.show()


In [ ]:
# 1. Through averaging of the Fourier transform determine the dominant frequency (center frequency)
# generated by the submarine. Verify your results through visualization.
avg_signal = np.zeros((N_grid, N_grid, N_grid))
for i in range(49):
  signalreshaped = np.reshape(d[:, i], (N_grid, N_grid, N_grid))
  aux = np.fft.fftshift(np.fft.fftn(signalreshaped))
  avg_signal += np.abs(aux)
avg_signal = avg_signal/49


In [ ]:
#Verify your results through visualization.
x1 = np.linspace(-10, 10, 64+1)
x = x1[0:64]
y, z = x, x
xx, yy, zz = np.meshgrid(x, y, z)
normal_avgsig_abs = np.abs(avg_signal)/np.abs(avg_signal).max()

avg_signal_data = go.Isosurface( x = xx.flatten(), y = yy.flatten(), z = zz.flatten(),
                                value = normal_avgsig_abs.flatten(), isomin = 0.7, isomax = 0.7)
clear_output(wait=True)
fig = go.Figure( data = avg_signal_data )
fig.show()


In [ ]:
#2. Design and implement a Filter to extract this center frequency in order to denoise the data and
# determine a more robust path of the submarine.
max = np.argmax(avg_signal)
shape = avg_signal.shape
max_location = np.unravel_index(max, shape)
ix0, iy0, iz0 = max_location
kx0, ky0, kz0 = K_grid[ix0], K_grid[iy0], K_grid[iz0]
kx, ky, kz = np.meshgrid(K_grid, K_grid, K_grid, indexing='ij')
sigma = 3
t = 2*sigma**2
gaussian_filter = np.exp(-((kx-kx0)**2 + (ky-ky0)**2 + (kz-kz0)**2) / t)

denoised = np.zeros_like(d)
for i in range(49):
    signal3D = np.reshape(d[:, i], (N_grid, N_grid, N_grid))
    fft_signal = np.fft.fftshift(np.fft.fftn(signal3D))
    filtered_signal = fft_signal * gaussian_filter
    denoised_signal = np.fft.ifftn(np.fft.ifftshift(filtered_signal))
    denoised_signal = np.real(denoised_signal)
    denoised[:, i] = denoised_signal.flatten()

path_x = np.zeros(49)
path_y = np.zeros(49)
path_z = np.zeros(49)
for i in range(49):
    signal = np.reshape(denoised[:, i], (N_grid, N_grid, N_grid))
    max_idx = np.argmax(np.abs(signal))
    ix, iy, iz = np.unravel_index(max_idx, signal.shape)
    path_x[i] = x[ix]
    path_y[i] = y[iy]
    path_z[i] = z[iz]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=path_x,y=path_y,z=path_z,
    mode='lines+markers'))
fig.show()


In [ ]:
# Location of the dominant frequency
print(kx0, ky0, kz0)

In [ ]:
import matplotlib.pyplot as plt
#3. Determine and plot the 𝑥, 𝑦 coordinates of the submarine path during the 24 hour period. This
#information can be used to deploy a sub-tracking aircraft to keep an eye on your submarine in the
#future.
plt.figure(figsize=(6,6))
plt.plot(path_x, path_y, '-o', markersize=3)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Submarine path in the xy plane (over 24 hours)')
plt.axis('equal')
plt.show()